[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc4_ml/corrections/seance2_correction.ipynb)

# Séance 4.2 — Prédire une décision — qui va résilier ?

**Correction** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- transformer des colonnes de texte en variables utilisables par un modèle
- ajuster une régression logistique et lire une probabilité de départ
- expliquer pourquoi la justesse est un piège sur des données déséquilibrées
- lire une matrice de confusion, la précision et le rappel
- choisir un seuil de décision à partir d'un coût, pas d'une habitude

## Correction

Solutions commentées. Comparez avec ce que vous aviez écrit : plusieurs formulations peuvent être correctes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics import precision_score, recall_score, roc_auc_score

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc4_ml/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
tel = pd.read_csv(BASE + "churn.csv")
tel["total"] = pd.to_numeric(tel["total"], errors="coerce")
tel = tel.dropna(subset=["total"])

y = tel["churn"]
X = pd.get_dummies(tel.drop(columns=["churn"]), drop_first=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(len(X_tr), "abonnes d'apprentissage,", len(X_te), "de test")

---

# Partie 1 — L'échauffement

Le code est déjà écrit : il ne reste que les `____` à remplir. Allez vite, l'essentiel
de la séance est dans la partie 2.

### Exercice 1 — Le nettoyage et le déséquilibre

> **Votre mission :**
> - Le nettoyage est fait dans la cellule de préparation.
> - Mettre le nombre d'abonnés conservés dans `n_abo`, et le taux de résiliation en % (1 décimale) dans `taux`.

In [ ]:
n_abo = len(tel)

# La moyenne d'une colonne de 0 et de 1, c'est la proportion de 1
taux = round(100 * tel["churn"].mean(), 1)

print(n_abo, "abonnes |", taux, "% de resiliations")

In [ ]:
verifier("1a - abonnes conserves", n_abo == 7032, "11 lignes ont ete ecartees")
verifier("1b - taux de resiliation", taux == 26.6, "mean() sur une colonne 0/1")

### Exercice 2 — Le contrat, déjà

> **Votre mission :**
> - Calculer le taux de résiliation par type de contrat, en % arrondi à 1 décimale → `par_contrat`.
> - Mettre celui des contrats mensuels dans `taux_mensuel`.

In [ ]:
par_contrat = (tel.groupby("contrat")["churn"].mean() * 100).round(1)
taux_mensuel = par_contrat["mensuel"]

print(par_contrat)

# 42,7 % contre 2,8 % : un facteur quinze entre le contrat mensuel et
# l'engagement deux ans.

In [ ]:
verifier("2 - churn des contrats mensuels", taux_mensuel == 42.7,
         "groupby('contrat') puis mean() sur churn")

### Exercice 3 — Du texte vers des colonnes

> **Votre mission :**
> - `X` est construit dans la préparation avec `get_dummies`.
> - Mettre son nombre de colonnes dans `nb_col`, et le nombre de colonnes du fichier d'origine dans `nb_col_avant`.

In [ ]:
nb_col_avant = tel.shape[1] - 1
nb_col = X.shape[1]

print(nb_col_avant, "colonnes ->", nb_col, "apres get_dummies")

In [ ]:
verifier("3a - colonnes avant", nb_col_avant == 9, "10 colonnes moins la cible churn")
verifier("3b - colonnes apres", nb_col == 14, "shape[1] donne le nombre de colonnes")

### Exercice 4 — Ajuster le modèle

> **Votre mission :**
> - Construire un pipeline `StandardScaler` puis `LogisticRegression(max_iter=1000)` → `m`, et l'ajuster sur l'apprentissage.
> - Récupérer les probabilités de départ du jeu de test → `proba`.

In [ ]:
m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
m.fit(X_tr, y_tr)

# predict_proba renvoie deux colonnes : [proba de 0, proba de 1].
# Celle qui nous interesse est la seconde, l'indice 1.
proba = m.predict_proba(X_te)[:, 1]
print(proba[:5].round(3))

In [ ]:
verifier("4a - nombre de probabilites", len(proba) == 2110, "on predit sur le jeu de test")
verifier("4b - ce sont bien des probabilites", proba.min() >= 0 and proba.max() <= 1,
         "la colonne d'indice 1 est la probabilite de depart")

### Exercice 5 — Le piège de la justesse

> **Votre mission :**
> - Calculer la justesse du modèle sur le test → `just_modele` (en %, 1 décimale).
> - Puis celle d'un modèle qui prédit que **personne** ne part → `just_nul`.
> - Combien de points le modèle gagne-t-il réellement ?

In [ ]:
pred = m.predict(X_te)

just_modele = round(100 * accuracy_score(y_te, pred), 1)

# La part de ceux qui restent : c'est la justesse du modele qui dit
# toujours "reste"
just_nul = round(100 * (1 - y_te.mean()), 1)
print(just_modele, "% contre", just_nul, "% ->", round(just_modele - just_nul, 1), "points")

In [ ]:
verifier("5a - justesse du modele", abs(just_modele - 79.8) < 0.5, "accuracy_score(y_te, pred)")
verifier("5b - justesse du modele nul", just_nul == 73.4,
         "c'est la proportion de clients qui restent")

### Exercice 6 — La matrice de confusion

> **Votre mission :**
> - Afficher la matrice de confusion du modèle.
> - Mettre dans `perdus` le nombre de clients **partis sans qu'on les ait détectés**.
> - C'est la case qui coûte de l'argent.

In [ ]:
mat = confusion_matrix(y_te, pred)
print(mat)

# Ligne 1 = ceux qui partent vraiment, colonne 0 = ceux qu'on predit
# comme restant. L'intersection : les partants qu'on n'a pas vus venir.
perdus = mat[1][0]
print(perdus, "clients perdus sans rien tenter")

In [ ]:
verifier("6 - clients perdus non detectes", abs(perdus - 255) <= 5,
         "ligne des vrais partants, colonne des predits restants")

### Exercice 7 — Précision et rappel

> **Votre mission :**
> - Calculer la précision → `prec` et le rappel → `rapp` du modèle, arrondis à 3 décimales.
> - Traduire chacun en une phrase de gestion, en commentaire.

In [ ]:
prec = round(precision_score(y_te, pred), 3)
rapp = round(recall_score(y_te, pred), 3)

print("precision", prec, "| rappel", rapp)

# Precision 0,64 : sur 10 abonnes contactes, 6 allaient vraiment partir.
# Rappel 0,545 : nous retrouvons un partant sur deux, l'autre s'en va.

In [ ]:
verifier("7a - precision", abs(prec - 0.640) < 0.02, "precision_score")
verifier("7b - rappel", abs(rapp - 0.545) < 0.02, "la fonction s'appelle recall_score")

### Exercice 8 — Descendre le seuil

> **Votre mission :**
> - Décider à **0,30** au lieu de 0,50 → `pred30`.
> - Recalculer précision et rappel → `prec30` et `rapp30`.
> - Lequel des deux monte, lequel descend ?

In [ ]:
pred30 = (proba > 0.30).astype(int)

prec30 = round(precision_score(y_te, pred30), 3)
rapp30 = round(recall_score(y_te, pred30), 3)
print("seuil 0.30 -> precision", prec30, "| rappel", rapp30)

# Le rappel monte (on retrouve plus de partants), la precision descend
# (on contacte plus de gens pour rien). C'est toujours ce compromis.

In [ ]:
verifier("8a - rappel a 0,30", rapp30 > rapp, "un seuil plus bas retrouve plus de partants")
verifier("8b - precision a 0,30", prec30 < prec, "et contacte plus de monde pour rien")

### Exercice 9 — Le gain de la campagne

> **Votre mission :**
> - Un appel coûte **15 €**, un client retenu rapporte **300 €**, une relance en convainc **30 %**.
> - Calculer le gain au seuil 0,50 → `gain50`, puis au seuil 0,20 → `gain20`.

In [ ]:
def gain(seuil):
    p = (proba > seuil).astype(int)
    vrais = ((p == 1) & (y_te == 1)).sum()       # partants effectivement rattrapes
    return vrais * 0.30 * 300 - p.sum() * 15     # 15 euros par appel passe

gain50 = gain(0.50)
gain20 = gain(0.20)
print(round(gain50), "euros contre", round(gain20), "euros")

# 8 000 EUR d'ecart, pour un seul nombre change dans une comparaison.

In [ ]:
verifier("9a - gain au seuil 0,50", abs(gain50 - 20370) < 600, "le cout d'un appel est 15")
verifier("9b - gain au seuil 0,20", abs(gain20 - 28365) < 600, "meme calcul, seuil 0.20")

### Exercice 10 — Question de synthèse

> **Votre mission :**
> - Le directeur de la relation client vous demande combien d'appels prévoir et ce que ça rapporte.
> - Au seuil retenu de 0,20 : le nombre d'appels → `nb_appels`, et le gain par appel → `gain_appel` (arrondi à 2 décimales).
> - Puis rédigez votre recommandation en commentaire.

In [ ]:
p20 = (proba > 0.20).astype(int)

nb_appels = int(p20.sum())
gain_appel = round(gain20 / nb_appels, 2)
print(nb_appels, "appels |", gain_appel, "euros de gain par appel")

# Recommandation possible :
# "Sur 2 110 abonnes, le modele en designe 1 073 a rappeler. La campagne
#  rapporte 28 400 EUR nets, soit 26 EUR par appel passe. Le seuil de
#  0,20 est volontairement bas : nos appels coutent 15 EUR et un client
#  retenu en rapporte 300, il est donc rentable d'appeler large. Ce
#  reglage doit etre revu si le cout d'un appel augmente."

In [ ]:
verifier("10a - nombre d'appels", abs(nb_appels - 1073) < 40, "sum() compte les 1")
verifier("10b - gain par appel", abs(gain_appel - 26.4) < 2, "divisez le gain par le nombre d'appels")

---

# Partie 2 — Les questions

Ici, plus de trous : **la cellule sous chaque question est vide**, et c'est à vous
d'écrire le code en entier. C'est exactement ce qu'on vous demandera pour le projet
final, et ce que fait un analyste devant un fichier qu'il découvre.

Certaines questions utilisent une commande que le cours n'a pas montrée. Quand c'est le
cas, l'énoncé vous la donne — savoir se servir d'une commande qu'on vient de lire fait
partie du métier.

> 💡 Pas de vérification automatique dans cette partie. Affichez systématiquement votre
> résultat, et demandez-vous s'il est **plausible** avant de passer à la suite : c'est
> le seul contrôle dont vous disposerez en entreprise.

### Question 11 — La courbe du seuil

> **Votre mission :**
> - Pour des seuils de 0,05 à 0,95, calculer précision et rappel, puis tracer les deux courbes sur la même figure.
> - Où se croisent-elles ? Que représente ce point ?

In [ ]:
seuils = np.arange(0.05, 1.0, 0.05)
lignes = [{"seuil": s,
           "precision": precision_score(y_te, (proba > s).astype(int), zero_division=0),
           "rappel": recall_score(y_te, (proba > s).astype(int))}
          for s in seuils]

pd.DataFrame(lignes).set_index("seuil").plot(figsize=(7, 4))
plt.title("Precision et rappel selon le seuil")
plt.show()

# Elles se croisent vers 0,35. Ce point n'a aucune vertu particuliere :
# c'est celui ou l'on se trompe autant dans les deux sens, ce qui n'a
# d'interet que si les deux erreurs coutent la meme chose. Ici, non.

### Question 12 — La courbe du gain

> **Votre mission :**
> - Tracer le gain de la campagne en fonction du seuil, de 0,05 à 0,95.
> - Lire l'optimum sur la figure, et le retrouver par le calcul.

In [ ]:
seuils = np.arange(0.05, 1.0, 0.05)
gains = pd.Series([gain(s) for s in seuils], index=seuils.round(2))

gains.plot(figsize=(7, 4))
plt.title("Gain de la campagne selon le seuil")
plt.ylabel("euros")
plt.show()

print("optimum :", gains.idxmax(), "->", round(gains.max()), "euros")

# La courbe est plate autour de l'optimum : entre 0,15 et 0,25 le gain
# bouge peu. C'est une bonne nouvelle — la decision n'a pas besoin d'etre
# reglee au millieme pres.

### Question 13 — Quand l'appel coûte plus cher

> **Votre mission :**
> - Refaire le calcul du gain avec un appel à **60 €** au lieu de 15 € — une visite commerciale plutôt qu'un appel.
> - Où passe l'optimum ? Qu'est-ce que ça dit du réglage d'un modèle ?

In [ ]:
def gain_cher(seuil, cout):
    p = (proba > seuil).astype(int)
    vrais = ((p == 1) & (y_te == 1)).sum()
    return vrais * 0.30 * 300 - p.sum() * cout

for cout in [15, 60]:
    g = pd.Series([gain_cher(s, cout) for s in np.arange(0.05, 1.0, 0.05)],
                  index=np.arange(0.05, 1.0, 0.05).round(2))
    print(f"cout {cout} EUR -> seuil optimal {g.idxmax()}, gain {round(g.max())} EUR")

# Quand le contact coute cher, il faut etre plus selectif : le seuil
# optimal remonte. Le meme modele, les memes probabilites, et pourtant
# une decision differente. Le modele ne decide pas, il informe.

### Question 14 — Les coefficients de la logistique

> **Votre mission :**
> - Extraire les coefficients du modèle et les trier par valeur absolue décroissante.
> - Les variables ayant été mises à l'échelle, ils sont comparables entre eux.
> - Quelles sont les trois qui pèsent le plus, et dans quel sens ?
> - *Nouveau :* dans un pipeline, on accède à la dernière étape par `m[-1]`.

In [ ]:
coefs = pd.Series(m[-1].coef_[0], index=X.columns)

coefs.reindex(coefs.abs().sort_values(ascending=False).index).round(3).head(6)

# Un coefficient positif pousse au depart, negatif retient. On retrouve
# le contrat en tete — la seance 4.3 confirmera avec d'autres methodes.

### Question 15 — L'AUC ne dépend pas du seuil

> **Votre mission :**
> - Vérifier que l'AUC est identique quel que soit le seuil, alors que la justesse, elle, change.
> - Pourquoi cette propriété rend-elle l'AUC pratique pour **comparer deux modèles** ?

In [ ]:
for s in [0.2, 0.5, 0.8]:
    p = (proba > s).astype(int)
    print(f"seuil {s} : justesse {accuracy_score(y_te, p):.3f} | "
          f"AUC {roc_auc_score(y_te, proba):.3f}")

# L'AUC se calcule sur les PROBABILITES, pas sur les decisions : elle
# mesure la capacite du modele a classer les partants avant les autres,
# independamment de l'endroit ou l'on coupe. C'est donc la bonne mesure
# pour comparer deux modeles — et la mauvaise pour decider d'une action.

### Question 16 — Un modèle plus simple fait-il pire ?

> **Votre mission :**
> - Ajuster un second modèle avec **trois variables seulement** : `anc`, `mensuel` et le contrat.
> - Comparer son AUC à celle du modèle complet.
> - La différence justifie-t-elle de collecter les onze autres colonnes ?

In [ ]:
petit = pd.get_dummies(tel[["anc", "mensuel", "contrat"]], drop_first=True)
a_tr, a_te, b_tr, b_te = train_test_split(
    petit, y, test_size=0.3, random_state=42, stratify=y)

simple = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
simple.fit(a_tr, b_tr)

print("modele complet :", round(roc_auc_score(y_te, proba), 3))
print("modele a 3 var :", round(roc_auc_score(b_te, simple.predict_proba(a_te)[:, 1]), 3))

# L'ecart est de quelques milliemes. Trois variables suffisent a
# reproduire l'essentiel : question a poser AVANT de lancer un chantier
# de collecte de donnees.

### Question 17 — Le déséquilibre, traité autrement

> **Votre mission :**
> - Réajuster la logistique avec `class_weight='balanced'`, qui donne plus de poids à la classe rare.
> - Comparer justesse, précision et rappel au modèle d'origine.
> - En quoi cet effet ressemble-t-il à celui du seuil ?

In [ ]:
equilibre = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced")).fit(X_tr, y_tr)
pe = equilibre.predict(X_te)

for nom, q in {"origine": pred, "equilibre": pe}.items():
    print(f"{nom:<10} justesse {accuracy_score(y_te, q):.3f}  "
          f"precision {precision_score(y_te, q):.3f}  rappel {recall_score(y_te, q):.3f}")

# La justesse BAISSE, le rappel monte fortement. C'est le meme arbitrage
# qu'un seuil abaisse, obtenu cette fois pendant l'apprentissage. Deux
# chemins, une seule question : quelle erreur coute le plus cher ?

### Question 18 — Qui sont les faux positifs ?

> **Votre mission :**
> - Isoler les abonnés que le modèle prédit partants **à tort**.
> - Comparer leur profil (ancienneté, facture, contrat) à celui des vrais partants.
> - Ces appels sont-ils vraiment perdus ?

In [ ]:
test = X_te.copy()
test["reel"] = y_te
test["predit"] = pred

faux = test.query("predit == 1 and reel == 0")
vrais = test.query("predit == 1 and reel == 1")

print("faux positifs :", len(faux), "| vrais positifs :", len(vrais))
pd.DataFrame({"faux alerte": faux[["anc", "mensuel"]].median(),
              "vrais partants": vrais[["anc", "mensuel"]].median()}).round(1)

# Les faux positifs ressemblent beaucoup aux vrais partants : jeunes
# abonnes, facture elevee, contrat mensuel. Ce ne sont pas des appels
# perdus, ce sont des clients A RISQUE qui ne sont pas encore partis.

### Question 19 — Et si on prédisait à l'envers ?

> **Votre mission :**
> - Inverser la cible : prédire `1 - churn`, c'est-à-dire « ce client reste-t-il ? ».
> - Comparer justesse, précision et rappel à ceux du modèle d'origine.
> - Qu'est-ce que ça montre sur ces trois mesures ?

In [ ]:
inverse = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
inverse.fit(X_tr, 1 - y_tr)
pi = inverse.predict(X_te)

print("justesse  :", round(accuracy_score(1 - y_te, pi), 3), "(identique)")
print("precision :", round(precision_score(1 - y_te, pi), 3))
print("rappel    :", round(recall_score(1 - y_te, pi), 3))

# La justesse ne bouge pas : elle est symetrique. Precision et rappel,
# eux, changent du tout au tout — ils sont definis PAR RAPPORT A LA
# CLASSE POSITIVE. Toujours preciser de quelle classe on parle.

### Question 20 — Question de synthèse

> **Votre mission :**
> - Le comité hésite : faut-il investir dans un meilleur modèle, ou dans un meilleur ciblage ?
> - Chiffrez les deux pistes : le gain obtenu en passant du seuil 0,50 au seuil optimal, et celui qu'apporterait un modèle parfait (rappel de 1 sans faux positifs) au seuil 0,50.
> - Puis tranchez, en commentaire.

In [ ]:
gains = pd.Series([gain(s) for s in np.arange(0.05, 1.0, 0.05)],
                  index=np.arange(0.05, 1.0, 0.05).round(2))

partants = int(y_te.sum())
parfait = partants * 0.30 * 300 - partants * 15

print("seuil 0,50           :", round(gain(0.50)), "euros")
print("meilleur seuil       :", round(gains.max()), "euros")
print("modele parfait       :", round(parfait), "euros")

# Regler le seuil rapporte 8 000 EUR et coute une apres-midi. Un modele
# PARFAIT — inatteignable — plafonnerait a 42 000 EUR. L'essentiel du
# gain disponible s'obtient donc par le ciblage, pas par l'algorithme.
# C'est la reponse a donner au comite.